# 📗 Sesión 3 · Ejercicio A — Estudiante
## Tarjeta KPI de bienestar laboral con semáforo de satisfacción

**Modalidad:** Guiado — KPI base dado, personaliza los umbrales, colores y CSS  
**Tiempo:** 35 minutos  
**Dataset:** IBM HR Analytics (`hr_ibm.csv`)

---

### Objetivo

Agregar una **quinta tarjeta KPI** a la app del módulo que muestre el promedio de
`WorkLifeBalance` con semáforo propio. También modificarás los colores CSS de la app.

### WorkLifeBalance en el dataset

La columna `WorkLifeBalance` tiene valores de **1 a 4**:
- 1 = Bad, 2 = Good, 3 = Better, 4 = Best

### Tu tarea

| Parte | Qué hacer |
|-------|-----------|
| **CSS** | Cambiar el color de la navbar a uno de tu elección |
| **KPI** | Crear `kpi_balance` con semáforo basado en la escala 1–4 |
| **Semáforo** | Rojo < 2.5 · Naranja 2.5–3.2 · Verde > 3.2 |

El KPI de rotación (`kpi_rotacion`) ya está dado — úsalo como modelo exacto.


## Celda 1 — Instalación y exploración

In [ ]:
# Instalación y exploración de WorkLifeBalance
import pandas as pd
import plotly.express as px
from shiny import App, ui, render, reactive

df = pd.read_csv('../hr_ibm.csv')

print("Vamos a explorar el indicador WorkLifeBalance")
print(f"Escala: {df['WorkLifeBalance'].min()} - {df['WorkLifeBalance'].max()}")
print(f"Promedio Global: {df['WorkLifeBalance'].mean():.2f}")

print()
print("Distribución")
print(df["WorkLifeBalance"].value_counts().sort_index().to_string())

print()
print("Promedio por departamento")
print(df.groupby("Department")["WorkLifeBalance"].mean().round(2).to_string())

print()
print("Promedio de Rotación")
print(df.groupby("Attrition")["WorkLifeBalance"].mean().round(2).to_string())


## Referencia: el KPI de rotación (úsalo como modelo)

```python
@render.ui
def kpi_rotacion():
    d = datos_filtrados()
    pct = (d["Attrition"] == "Yes").mean() * 100
    # Semáforo: rojo > 20%, naranja 10–20%, verde <= 10%
    color = "#e71d27" if pct > 20 else ("#d2690e" if pct > 10 else "#2E7D32")
    return ui.div(
        ui.div(f"{pct:.1f}%", class_="kpi-valor", style=f"color:{color}"),
        ui.div("Tasa de rotación", class_="kpi-label"),
        class_="kpi-card"
    )
```

Para `kpi_balance` necesitas:
- Calcular `bal = d["WorkLifeBalance"].mean()`
- Semáforo **inverso**: rojo si `bal < 2.5`, naranja si `bal <= 3.2`, verde si `bal > 3.2`
- Mostrar el valor como `f"{bal:.2f}/4"`
- Cambiar la etiqueta a `"Balance vida–trabajo"`


## Celda 3 — App con huecos

In [ ]:
import pandas as pd
import plotly.express as px
from shiny import App, ui, render, reactive

df = pd.read_csv('../hr_ibm.csv')

#nuestros filtros
opts_dept = ["Todos"] + sorted(df["Department"].unique().tolist())
opst_attr = ["Todos", "Yes", "No"]

#Lo primero que necesitamos es definir la css
css = ui.tags.style("""
    body { background: #F0F4F8; font-family: Segoe UI, Arial, sans-serif;  }
    .navbar { background-color:#2C3E7A!important;  }
    .kpi_card { background.color:white; border-radius: 2px; 
    text-alig:center; box-shadow: 0 2px rgba(0,0,0,0.1); padding:20px; }
    .navbar-brand, nav-link { color:white!important; font-weight:500; }
    .kpi_valor { font-size:2em; font-weight:bold;  }
    .kpi-label { font-size:.85em; color:#666; margin-top:4px; } 
""")

app_ui = ui.page_navbar(
    ui.nav_panel("Resumen", ui.layout_sidebar(
        ui.sidebar(
            ui.h5("Filtros"),
            ui.input_select("dept", "Detarmento: ", choices=opts_dept),
            ui.input_select("attrition", "Rotación", choices=opst_attr),
            ui.input_slider("edad", "Rango de Edad", min=18, max=60, value=[18, 60]),
            ui.input_action_button("btn_reset", "Restablecer", class_="btn-sm mt-2")
        ),
        ui.h4("Indicadores clave"),
        ui.row(
            ui.column(2, ui.output_ui("kpi_total")),
            ui.column(2, ui.output_ui("kpi_rotación")),
            ui.column(2, ui.output_ui("kpi_salario")),
            ui.column(2, ui.output_ui("kpi_depto")),
            ui.column(3, ui.output_ui("kpi_balance")),
        ),
        ui.h4("Visualizaciones"),
        ui.row(
            ui.column(6, ui.output_ui("grafica_rotacion")),
            ui.column(6, ui.output_ui("grafica_scatter")),
        ),
    )),
    ui.nav_panel("Detalle", ui.output_table("table_detalle")),
    title="Dasboard",
    header=css
)


def server(input, output, session):

    @reactive.calc
    def datos_filtrados():
        d = df.copy()
        if input.dept() != "Todos": d= d[d["Department"] == input.dept()]
        if input.attrition() != "Todos": d= d[d["Attrition"] == input.attrition()]
        emin, emax = input.edad()
        return d[(d["Age"] >= emin) & (d["Age"] >= emax) ]

    @reactive.effect
    @reactive.event(input.btn_reset)
    def _reset():
        ui.update_select("dept", selected="Todos")
        ui.update_select("attrition", selected="Todos")
        ui.update_slider("edad", value=[18, 60])

    @render.ui
    def kpi_total():
        n = len(datos_filtrados())
        return ui.div(
            ui.div(f"{n:, }", class_="kpi_valor", style="color:#1F3864"),
            ui.div("Total empleados", class_="kpi-label"), class_="kpi-card"
            )

    @render.ui
    def kpi_rotacion():
        d = datos_filtrados()
        pct = (d["Attrition"] == "Yes").mean()*100
        col = "#C62828" if pct > 20 else ("#F57F17" if pct > 10 else "#2E7D32")
        return ui.div(
            ui.div(f"{pct:.1f}%", class_="kpi-valor", style=f"color:{col}"),
            ui.div("Tasa de rotación: ", class_="kpi-label"), class_="kpi-card"
        )

    @render.ui
    def kpi_salario():
        sal = datos_filtrados()["MonthlyIncome"].mean()
        return ui.div(
            ui.div(f"${sal:,.0f}", class_="kpi-valor", style="color:#2E7D32"),
            ui.div("Salario Promedio ", class_="kpi-label"), class_="kpi-card"
        )

    @render.ui
    def kpi_depto():
        d=datos_filtrados()
        if len(d) == 0:
            return ui.div(
                ui.div("-", class_="kpi-valor", style="color:#999"),
                ui.div("Mayor Rotación", class_="kpi-label"), class_="kpi-card"
            )
        dept = d.groupby("Department")["Attrition"].apply(
            lambda x : (x == "Yes").mean()
        ).idxmax()
        pct = d[d["Department"] == dept]["Attrition"].eq("Yes").mean()*100
        return ui.div(
            ui.div(dept, class_="kpi-valor", style="color:#E65100; font-size:1em"),
            ui.div(f"Mayor rotación: {pct:.1f}", class_="kpi-label"), class_="kpi-card"
            )

    @render.ui
    def kpi_balance():
        d = datos_filtrados()
        bal = d["WorkLifeBalance"].mean()
        col = "#2E7D32" if bal > 3.2 else ("#F57F17" if bal >= 2.5 else "#C62828")
        return ui.div(
            ui.div(f"{bal:.2f}/4", class_="kpi-valor", style=f"color:{col}"),
            class_="kpi_card"
        )

    @render.ui
    def grafica_scatter():
        d = datos_filtrados()
        fig = px.scatter(d, x="MonthlyIncome", y="JobSatisfaction", color="Attrition", color_discrete_map={"Yes" : "#E53935", "No": "#43a047"}, hover_data=["JobRole", "Age"], opacity=0.7, title="Salario VS Satisfacción de Vida")
        fig.update_layout(height=320)
        return ui.HTML(fig.to_html(include_plotlyjs="cdn"))

    @render.ui
    def grafica_rotacion():
        d = datos_filtrados()
        rot = d.groupby("Department")["Attrition"].apply(
            lambda x : (x == "Yes").mean()*100
        ).reset_index(name="Rotacion_pct").round(1)
        fig = px.bar(rot, x="Department", y="Rotacion_pct", color="Department", text_auto=".1f", title="Rotación por departamento (%)")
        fig.update_layout(showlegend=False, height=320)
        return ui.HTML(fig.to_html(include_plotlyjs="cdn"))

    @render.ui
    def tabla_detalle():
        cols = ["EmployeeNumber", "Department", "JobRole", "Age", "Attrition", "MonthlyIncome", "WorkLifeBalance"]
        return datos_filtrados()[cols].head(20)

app = App(app_ui, server)
# shiny run app.py — o ejecutar app.run() si el entorno lo soporta


import threading

def _iniciar():
    app.run(host="127.0.0.1", port=8000)

hilo = threading.Thread(target=_iniciar, daemon=True)
hilo.start()
print("App corriendo en http://127.0.0.1:8000")
    




App corriendo en http://127.0.0.1:8000


INFO:     Started server process [27980]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:39811 - "GET / HTTP/1.1" 200 OK


INFO:     127.0.0.1:16545 - "WebSocket /websocket/" [accepted]
INFO:     connection open
Traceback (most recent call last):
  File "C:\Users\demon\AppData\Roaming\Python\Python312\site-packages\shiny\session\_session.py", line 2261, in output_obs
    value = await renderer.render()
            ^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\demon\AppData\Roaming\Python\Python312\site-packages\shiny\render\renderer\_renderer.py", line 276, in render
    value = await self.fn()
            ^^^^^^^^^^^^^^^
  File "C:\Users\demon\AppData\Roaming\Python\Python312\site-packages\shiny\render\renderer\_renderer.py", line 372, in __call__
    return await self._fn()
           ^^^^^^^^^^^^^^^^
  File "C:\Users\demon\AppData\Roaming\Python\Python312\site-packages\shiny\_utils.py", line 277, in fn_async
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\demon\AppData\Local\Temp\ipykernel_27980\2300776998.py", line 72, in kpi_total
    ui.div(f"{n:, }", class_="kpi_valor", sty

## Celda 4 — Preguntas de análisis

In [ ]:
# Preguntas de análisis — usa los filtros de tu app para responder

print("Preguntas de análisis — Ejercicio 3A")
print("=" * 45)
print()
print("P1: Deja todos los filtros en Todos.")
print("    Observa el KPI de rotación y el KPI de balance.")
print("    Ahora filtra por Departamento = Human Resources.")
print("    ¿El departamento con mayor rotación también tiene el KPI de balance más bajo?")
print()
print("P2: Filtra por Rotación = Yes (empleados que se fueron).")
print("    ¿El KPI de balance es mayor o menor que sin filtro?")
print("    ¿Qué implicación tiene eso para los gestores de RH?")
print()
print("P3: Explora distintos departamentos con el selector.")
print("    ¿En qué departamento el KPI de balance aparece en VERDE?")
print("    ¿Ese departamento también tiene baja rotación?")
print()
print("Escribe tus respuestas (convierte a Markdown):")
print("P1: ...")
print("P2: ...")
print("P3: ...")
